# Notebook 07 — Monitoring & Decision Framework
## Putting It All Together

**Objective**: Use `ACCOUNT_USAGE` views to monitor performance, and apply the decision framework to choose the right optimization.

| Feature | Best Use Case (Banking) | Worst Use Case | Cost |
|---|---|---|---|
| Natural ordering | Monthly statements (by date) | Customer lookups (by account_id) | Free |
| Clustering | Customer-level queries on TRANSACTIONS | Clustering on `currency_code` (3 values) | Serverless |
| SOS | Fraud: transaction_id lookup, merchant substring | Broad `amount > 100` filter | Serverless + storage |
| QAS | Ad-hoc fraud analytics with large scans | Simple PK lookups already <1s | Serverless |
| Warehouse sizing | Match size to query complexity | 4XL for trivial lookups | Per-second |

In [ ]:
USE WAREHOUSE HOL_M;
USE SCHEMA JPMC_PERF_HOL.CONSUMER_BANKING;

---
## Step 1: Identify Slowest Queries

In [ ]:
-- Top 10 slowest queries from this HOL session
SELECT
    query_id,
    SUBSTR(query_text, 1, 80) AS query_preview,
    warehouse_name,
    warehouse_size,
    total_elapsed_time / 1000 AS elapsed_sec,
    partitions_scanned,
    partitions_total,
    bytes_spilled_to_remote_storage / (1024*1024*1024) AS spill_remote_gb
FROM SNOWFLAKE.ACCOUNT_USAGE.QUERY_HISTORY
WHERE database_name = 'JPMC_PERF_HOL'
  AND start_time > DATEADD(hour, -2, CURRENT_TIMESTAMP())
  AND query_type = 'SELECT'
ORDER BY total_elapsed_time DESC
LIMIT 10;

---
## Step 2: Identify Queries That Scan Too Many Partitions

In [ ]:
-- Queries scanning >80% of partitions — candidates for clustering or SOS
SELECT
    query_id,
    SUBSTR(query_text, 1, 80) AS query_preview,
    partitions_scanned,
    partitions_total,
    ROUND(partitions_scanned / NULLIF(partitions_total, 0) * 100, 1) AS pct_scanned,
    total_elapsed_time / 1000 AS elapsed_sec
FROM SNOWFLAKE.ACCOUNT_USAGE.QUERY_HISTORY
WHERE database_name = 'JPMC_PERF_HOL'
  AND start_time > DATEADD(hour, -2, CURRENT_TIMESTAMP())
  AND partitions_total > 100
  AND partitions_scanned / NULLIF(partitions_total, 0) > 0.8
ORDER BY total_elapsed_time DESC
LIMIT 10;

---
## Step 3: Identify Queries That Spill

In [ ]:
-- Queries spilling to storage — candidates for warehouse upsizing
SELECT
    query_id,
    SUBSTR(query_text, 1, 80) AS query_preview,
    warehouse_size,
    bytes_spilled_to_local_storage / (1024*1024*1024) AS spill_local_gb,
    bytes_spilled_to_remote_storage / (1024*1024*1024) AS spill_remote_gb,
    total_elapsed_time / 1000 AS elapsed_sec
FROM SNOWFLAKE.ACCOUNT_USAGE.QUERY_HISTORY
WHERE database_name = 'JPMC_PERF_HOL'
  AND start_time > DATEADD(hour, -2, CURRENT_TIMESTAMP())
  AND (bytes_spilled_to_local_storage > 0 OR bytes_spilled_to_remote_storage > 0)
ORDER BY bytes_spilled_to_remote_storage DESC
LIMIT 10;

---
## Decision Tree

```
Query is slow. Why?
│
├── Scanning too many partitions?
│   ├── Filter aligns with load order? → Already optimized (natural clustering)
│   ├── Point lookup / substring?     → Search Optimization Service
│   └── Range query on non-ordered col? → Automatic Clustering
│
├── Spilling to storage?
│   └── Size up the warehouse (M → L)
│
├── Large scan with selective filter?
│   └── Enable Query Acceleration Service
│
└── None of the above?
    └── Check query pattern: SELECT *, late filters, cache-defeating functions
```

---
## Step 4: Cost Summary

In [ ]:
-- Credits consumed during this HOL by warehouse
SELECT
    warehouse_name,
    SUM(credits_used) AS total_credits
FROM SNOWFLAKE.ACCOUNT_USAGE.WAREHOUSE_METERING_HISTORY
WHERE start_time > DATEADD(hour, -3, CURRENT_TIMESTAMP())
  AND warehouse_name LIKE 'HOL%'
GROUP BY 1
ORDER BY total_credits DESC;

---
## Summary: Performance Engineering Principles

1. **Measure first** — Never optimize blind. Use `QUERY_HISTORY` and Query Profile.
2. **Free wins first** — Natural ordering, projection pruning, result caching cost nothing.
3. **Right-size warehouses** — Spilling = size up. No spilling on XS = don't size up.
4. **Clustering for repeated patterns** — Only when natural order doesn't match your queries.
5. **SOS for needles in haystacks** — Point lookups and substring searches only.
6. **QAS for ad-hoc analytics** — Large scans with selective filters.
7. **Always estimate cost** — Use `SYSTEM$ESTIMATE_*` functions before enabling features.

---
## Cleanup (optional)

In [ ]:
-- Uncomment to tear down the entire HOL environment
-- DROP DATABASE IF EXISTS JPMC_PERF_HOL;
-- DROP WAREHOUSE IF EXISTS HOL_XS;
-- DROP WAREHOUSE IF EXISTS HOL_M;
-- DROP WAREHOUSE IF EXISTS HOL_L;
-- DROP WAREHOUSE IF EXISTS HOL_NO_QAS;
-- DROP WAREHOUSE IF EXISTS HOL_QAS;